<a href="https://colab.research.google.com/github/Lydia-fadele/AfroDiabDB/blob/main/notebooks/AfroDiabDB_v1.2_Ch2_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AfroDiabDB v1.2: Computational Curation & Descriptor Pipeline

An automated cheminformatics workflow for structure standardization, PubChem API verification, salt stripping, molecular descriptor calculations, drug-likeness filtering, and library consolidation.

In [ ]:

# Setup Environment Dependencies
!pip install rdkit pubchempy openpyxl pandas -q
print("✓ Environment successfully initialized.")

## Phase 1: Data Standardization & PubChem API Integration
- Standardizes dataset column headers across records.
- Performs internal propagation (forward and backward filling) of PubChem CIDs, SMILES, and formulas within compound groups.
- Queries the PubChem REST API via `pubchempy` to retrieve missing structural attributes for unannotated compounds.

In [ ]:

import pandas as pd
import pubchempy as pcp
from google.colab import files

print("Please upload 'AfroDiabDB_v1.2.xlsx' (or your raw dataset file):")
uploaded = files.upload()
raw_file = list(uploaded.keys())[0]

df = pd.read_excel(raw_file)
df.columns = df.columns.str.strip()
print(f"\n✓ Loaded dataset '{raw_file}' with {len(df)} total records.")

# Propagate existing structural fields internally
cols_to_fill = ['PubChem_CID', 'Canonical_SMILES', 'Molecular_Formula']
for col in cols_to_fill:
    if col in df.columns:
        df[col] = df.groupby(df['Compound_Name'].str.strip().str.lower())[col].transform(lambda x: x.ffill().bfill())

# Query PubChem API for unpopulated entities
missing_mask = df['PubChem_CID'].isna()
unique_missing = df.loc[missing_mask, 'Compound_Name'].unique()
print(f"[Phase 1] Querying PubChem API for {len(unique_missing)} missing compounds...")

pubchem_cache = {}
for compound in unique_missing:
    try:
        results = pcp.get_compounds(compound, 'name')
        if results:
            c = results[0]
            pubchem_cache[compound] = {
                'PubChem_CID': c.cid,
                'Canonical_SMILES': c.canonical_smiles,
                'Molecular_Formula': c.molecular_formula
            }
    except Exception:
        pass

for index, row in df[missing_mask].iterrows():
    name = row['Compound_Name']
    if name in pubchem_cache:
        df.loc[index, 'PubChem_CID'] = pubchem_cache[name]['PubChem_CID']
        df.loc[index, 'Canonical_SMILES'] = pubchem_cache[name]['Canonical_SMILES']
        df.loc[index, 'Molecular_Formula'] = pubchem_cache[name]['Molecular_Formula']

output_phase1 = 'AfroDiabDB_v1.2_Phase1_Standardized.xlsx'
df.to_excel(output_phase1, index=False)
print(f"✓ Phase 1 complete. Saved output as '{output_phase1}'.")

## Phase 2: Structure Validation & Salt Stripping
- Validates canonical SMILES parsing using **RDKit**.
- Strips counter-ions, solvents, and salt adducts via RDKit `SaltRemover`.
- Uses PubChem CIDs as an automated fallback to repair unparseable SMILES strings.
- Deduplicates unique compound-plant records to ensure data integrity.

In [ ]:

from rdkit import Chem, RDLogger
from rdkit.Chem import SaltRemover

RDLogger.DisableLog('rdApp.*')
remover = SaltRemover.SaltRemover()

df_valid = df.dropna(subset=['PubChem_CID']).copy()
df_valid = df_valid[df_valid['Compound_Name'] != 'Compound_Name'].copy()

def verify_and_repair_smiles(row):
    smiles = str(row['Canonical_SMILES']).strip() if pd.notna(row['Canonical_SMILES']) else ""
    cid_val = row['PubChem_CID']

    mol = Chem.MolFromSmiles(smiles) if smiles else None

    # Repair missing or broken SMILES via PubChem CID
    if mol is None and pd.notna(cid_val):
        try:
            cid_int = int(float(cid_val))
            c = pcp.Compound.from_cid(cid_int)
            smiles = c.canonical_smiles
            mol = Chem.MolFromSmiles(smiles)
        except Exception:
            pass

    if mol is None:
        return None

    stripped_mol = remover.StripMol(mol)
    return Chem.MolToSmiles(stripped_mol, canonical=True)

df_valid['Canonical_SMILES_Verified'] = df_valid.apply(verify_and_repair_smiles, axis=1)
df_verified = df_valid.dropna(subset=['Canonical_SMILES_Verified']).copy()
df_final_verified = df_verified.drop_duplicates(subset=['Compound_Name', 'Plant_Name'], keep='first').copy()

output_phase2 = 'AfroDiabDB_v1.2_Phase2_Verified_Structures.xlsx'
df_final_verified.to_excel(output_phase2, index=False)
print(f"✓ Phase 2 complete. Retained {len(df_final_verified)} verified authentic structures.")

## Phase 3: Molecular Descriptor Calculation
Computes 12 essential physicochemical, structural, and drug-likeness molecular descriptors using **RDKit**:
* **Basic Physicochemical**: Molecular Weight ($MW$), $\log P$, Hydrogen Bond Acceptors ($HBA$), Hydrogen Bond Donors ($HBD$), Topological Polar Surface Area ($TPSA$), Rotatable Bonds ($RB$).
* **Extended Structural & Drug-Likeness**: Fraction $CSP3$, Ring Count, Aromatic Ring Count, Heavy Atom Count, Molar Refractivity ($MR$), Quantitative Estimate of Drug-likeness ($QED$).

In [ ]:
from rdkit.Chem import Descriptors, Lipinski, QED, rdMolDescriptors, Crippen

def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return pd.Series([None] * 12)
    return pd.Series([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Lipinski.NumHAcceptors(mol),
        Lipinski.NumHDonors(mol),
        Descriptors.TPSA(mol),
        Lipinski.NumRotatableBonds(mol),
        rdMolDescriptors.CalcFractionCSP3(mol),
        rdMolDescriptors.CalcNumRings(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        mol.GetNumHeavyAtoms(),
        Crippen.MolMR(mol),
        QED.qed(mol)
    ])

descriptor_names = [
    'Molecular_Weight', 'LogP', 'HBA', 'HBD', 'TPSA', 'Rotatable_Bonds',
    'Fraction_CSP3', 'Ring_Count', 'Aromatic_Ring_Count', 'Heavy_Atom_Count',
    'Molar_Refractivity', 'QED_Score'
]

df_final_verified[descriptor_names] = df_final_verified['Canonical_SMILES_Verified'].apply(compute_descriptors)

output_phase3 = 'AfroDiabDB_v1.2_Phase3_Descriptors.xlsx'
df_final_verified.to_excel(output_phase3, index=False)
print(f"✓ Phase 3 complete. Calculated {len(descriptor_names)} molecular descriptors.")

## Phase 4 & 5: Drug-Likeness Evaluation & Library Consolidation
1. Assesses lead compound viability across Lipinski, Veber, and Ghose rules.
2. Merges `AfroDiabDB v1.1` and `v1.2` entries into a single unique master library.
3. Exports final output to Google Drive and local storage.

In [ ]:

from google.colab import drive

# Drug-Likeness Rules
df_clean = df_final_verified.copy()
df_clean['Lipinski_Violations'] = df_clean.apply(
    lambda r: sum([r['Molecular_Weight'] > 500, r['LogP'] > 5, r['HBA'] > 10, r['HBD'] > 5]), axis=1
)
df_clean['Lipinski_Pass'] = df_clean['Lipinski_Violations'] <= 1
df_clean['Veber_Pass'] = (df_clean['TPSA'] <= 140) & (df_clean['Rotatable_Bonds'] <= 10)
df_clean['Ghose_Pass'] = (df_clean['Molecular_Weight'].between(160, 480)) & \
                        (df_clean['LogP'].between(-0.4, 5.6)) & \
                        (df_clean['Molar_Refractivity'].between(40, 130)) & \
                        (df_clean['Heavy_Atom_Count'].between(20, 70))

print("=== Drug-Likeness Assessment Summary ===")
print(f"Lipinski Pass: {df_clean['Lipinski_Pass'].sum()}/{len(df_clean)} ({df_clean['Lipinski_Pass'].mean()*100:.1f}%)")
print(f"Veber Pass:    {df_clean['Veber_Pass'].sum()}/{len(df_clean)} ({df_clean['Veber_Pass'].mean()*100:.1f}%)")
print(f"Ghose Pass:    {df_clean['Ghose_Pass'].sum()}/{len(df_clean)} ({df_clean['Ghose_Pass'].mean()*100:.1f}%)")

# Upload v1.1 to consolidate
print("\nPlease upload 'AfroDiabDB_v1.1_Unique_Library.xlsx' (or CSV):")
uploaded_v11 = files.upload()
file_v11_name = list(uploaded_v11.keys())[0]

try:
    df_v11 = pd.read_excel(file_v11_name)
except Exception:
    df_v11 = pd.read_csv(file_v11_name)

df_combined = pd.concat([df_v11, df_clean], ignore_index=True)
df_combined['Compound_Name_Clean'] = df_combined['Compound_Name'].astype(str).str.strip().str.lower()

df_unique = df_combined.drop_duplicates(subset=['Compound_Name_Clean', 'Plant_Name'], keep='last').copy()
df_unique.drop(columns=['Compound_Name_Clean'], inplace=True, errors='ignore')

# Save Master File
output_final = 'AfroDiabDB_v1.2_Unique_Library.xlsx'
df_unique.to_excel(output_final, index=False)

# Backup to Drive
try:
    drive.mount('/content/drive', force_remount=True)
    df_unique.to_excel(f'/content/drive/MyDrive/{output_final}', index=False)
    print(f"\n✓ Master library successfully consolidated ({len(df_unique)} unique records) and backed up to Google Drive!")
except Exception as e:
    print(f"\n✓ Master library saved locally ({len(df_unique)} unique records). Drive mount skipped.")

files.download(output_final)

## Phase 6, 7 & 8: Statistical Profiling, PCA Chemical Space Analysis & Data Visualization
- Computes library-wide descriptive statistics and superclass distributions.
- Performs Principal Component Analysis (PCA) comparing AfroDiabDB v1.2 against FDA-approved antidiabetic controls.
- Generates publication-quality charts (300 DPI) for manuscript inclusion.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen, rdMolDescriptors, QED
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.dpi': 300})

# ==========================================
# 1. DEFINE FDA ANTIDIABETIC CONTROLS (19 DRUGS)
# ==========================================
fda_data = [
    {"Compound_Name": "Metformin", "Class": "Biguanide", "Canonical_SMILES": "CN(C)C(=N)NC(=N)N"},
    {"Compound_Name": "Sitagliptin", "Class": "DPP-4 Inhibitor", "Canonical_SMILES": "NC(CC(=O)N1CC2CCC(C1)N2)Cc1cc(CC(F)(F)F)nc2cc(C(F)(F)F)nn12"},
    {"Compound_Name": "Saxagliptin", "Class": "DPP-4 Inhibitor", "Canonical_SMILES": "CC1(C2CC(C1)C(C2)(C#N)N)C(=O)NC(C31CC2CC(C1)CC(C2)O)C(=O)N"},
    {"Compound_Name": "Linagliptin", "Class": "DPP-4 Inhibitor", "Canonical_SMILES": "Cc1nc2c(n1Cc3nc(c4ccccc4n3)C)n(C)c(=O)n(CC#CC)c2=O"},
    {"Compound_Name": "Alogliptin", "Class": "DPP-4 Inhibitor", "Canonical_SMILES": "CC1=NC=C(C=N1)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)NC(=O)NC3CCCCC3"},
    {"Compound_Name": "Empagliflozin", "Class": "SGLT2 Inhibitor", "Canonical_SMILES": "CC1=CC=C(C=C1)C2=CC(=C(C=C2)Cl)C3C(C(C(C(O3)CO)O)O)O"},
    {"Compound_Name": "Dapagliflozin", "Class": "SGLT2 Inhibitor", "Canonical_SMILES": "CCOc1ccc(Cc2cc(C3OC(CO)C(O)C(O)C3O)ccc2Cl)cc1"},
    {"Compound_Name": "Canagliflozin", "Class": "SGLT2 Inhibitor", "Canonical_SMILES": "Cc1ccc(cc1)C2(C(C(C(O2)CO)O)O)c3ccc(cc3)Cc4ccc(s4)F"},
    {"Compound_Name": "Pioglitazone", "Class": "Thiazolidinedione", "Canonical_SMILES": "CCc1ccc(CCN2CCC(CC2)Oc3ccc(CC4SC(=O)NC4=O)cc3)nc1"},
    {"Compound_Name": "Rosiglitazone", "Class": "Thiazolidinedione", "Canonical_SMILES": "CN(CCOC1=CC=C(C=C1)CC2C(=O)NC(=O)S2)C3=NC=CC=C3"},
    {"Compound_Name": "Glibenclamide", "Class": "Sulfonylurea", "Canonical_SMILES": "COc1ccc(cc1C(=O)NCCc2ccc(cc2)S(=O)(=O)NC(=O)NC3CCCCC3)Cl"},
    {"Compound_Name": "Glimepiride", "Class": "Sulfonylurea", "Canonical_SMILES": "CCC1=C(C)C(=O)N(CCN2CCC(CC2)c3ccc(cc3)S(=O)(=O)NC(=O)NC4CCCCC4)C1=O"},
    {"Compound_Name": "Glipizide", "Class": "Sulfonylurea", "Canonical_SMILES": "CC1=NC=C(C=N1)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)NC(=O)NC3CCCCC3"},
    {"Compound_Name": "Gliclazide", "Class": "Sulfonylurea", "Canonical_SMILES": "CC1=CC=C(C=C1)S(=O)(=O)NC(=O)NN2CC3CCCC3C2"},
    {"Compound_Name": "Acarbose", "Class": "Alpha-Glucosidase Inhibitor", "Canonical_SMILES": "CC1C(C(C(C(O1)OC2C(OC(C(C2O)O)OC3C(OC(C(C3O)O)O)CO)CO)O)O)NC4C=C(C(C(C4O)O)O)CO"},
    {"Compound_Name": "Miglitol", "Class": "Alpha-Glucosidase Inhibitor", "Canonical_SMILES": "OCCN1CC(O)C(O)C(O)C1CO"},
    {"Compound_Name": "Voglibose", "Class": "Alpha-Glucosidase Inhibitor", "Canonical_SMILES": "OCC(O)CNC1C(O)C(O)C(O)C(O)C1O"},
    {"Compound_Name": "Repaglinide", "Class": "Meglitinide", "Canonical_SMILES": "CCCC1OCCC1c2cc(cc(c2)C(=O)O)N3CCCCC3"},
    {"Compound_Name": "Nateglinide", "Class": "Meglitinide", "Canonical_SMILES": "CC(C)C1CCC(CC1)C(=O)NC(CC2ccccc2)C(=O)O"}
]

df_fda = pd.DataFrame(fda_data)

# Compute Descriptors for FDA Controls
fda_descriptors = []
for idx, row in df_fda.iterrows():
    m = Chem.MolFromSmiles(row['Canonical_SMILES'])
    if m:
        fda_descriptors.append([
            Descriptors.MolWt(m), Descriptors.MolLogP(m), Lipinski.NumHAcceptors(m),
            Lipinski.NumHDonors(m), Descriptors.TPSA(m), Lipinski.NumRotatableBonds(m),
            rdMolDescriptors.CalcFractionCSP3(m), rdMolDescriptors.CalcNumRings(m),
            rdMolDescriptors.CalcNumAromaticRings(m), m.GetNumHeavyAtoms(),
            Crippen.MolMR(m), QED.qed(m)
        ])
    else:
        fda_descriptors.append([None]*12)

desc_cols = ['Molecular_Weight', 'LogP', 'HBA', 'HBD', 'TPSA', 'Rotatable_Bonds',
             'Fraction_CSP3', 'Ring_Count', 'Aromatic_Ring_Count', 'Heavy_Atom_Count',
             'Molar_Refractivity', 'QED_Score']

df_fda[desc_cols] = pd.DataFrame(fda_descriptors)

# ==========================================
# 2. PHASE 6: STATISTICAL SUMMARY
# ==========================================
print("================ PHASE 6: STATISTICAL SUMMARY ================")
print(f"✓ AfroDiabDB Records: {len(df_clean)}")
print(f"✓ FDA Benchmark Controls: {len(df_fda)}")

# ==========================================
# 3. PHASE 7: PCA CHEMICAL SPACE ANALYSIS
# ==========================================
print("\n================ PHASE 7: PCA CHEMICAL SPACE MAP ================")
scaler = StandardScaler()
X_phyto = scaler.fit_transform(df_clean[desc_cols].dropna())
X_fda = scaler.transform(df_fda[desc_cols].dropna())

pca = PCA(n_components=2)
coords_phyto = pca.fit_transform(X_phyto)
coords_fda = pca.transform(X_fda)

var_exp = pca.explained_variance_ratio_ * 100
print(f"PC1 Variance Explained: {var_exp[0]:.2f}%")
print(f"PC2 Variance Explained: {var_exp[1]:.2f}%")

# ==========================================
# 4. PHASE 8: PUBLICATION VISUALIZATIONS
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Fig A: MW vs LogP
sns.scatterplot(data=df_clean, x='Molecular_Weight', y='LogP', hue='Lipinski_Pass',
                palette={True: '#2ecc71', False: '#e74c3c'}, alpha=0.75, ax=axes[0, 0])
axes[0, 0].set_title('A: Molecular Weight vs. LogP (AfroDiabDB)', fontweight='bold')
axes[0, 0].axvline(500, color='grey', linestyle='--')
axes[0, 0].axhline(5, color='grey', linestyle='--')

# Fig B: Superclasses
top_classes = df_clean['Compound_Superclass'].value_counts().head(6)
sns.barplot(x=top_classes.values, y=top_classes.index, palette='crest', ax=axes[0, 1])
axes[0, 1].set_title('B: Primary Compound Superclasses', fontweight='bold')

# Fig C: Rule Compliance
pass_rates = [df_clean['Lipinski_Pass'].mean()*100, df_clean['Veber_Pass'].mean()*100, df_clean['Ghose_Pass'].mean()*100]
sns.barplot(x=['Lipinski Rule', 'Veber Filter', 'Ghose Filter'], y=pass_rates, palette='Blues_d', ax=axes[1, 0])
axes[1, 0].set_title('C: Drug-Likeness Rule Compliance (%)', fontweight='bold')
axes[1, 0].set_ylim(0, 105)
for i, v in enumerate(pass_rates):
    axes[1, 0].text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold')

# Fig D: PCA Chemical Space Map
axes[1, 1].scatter(coords_phyto[:, 0], coords_phyto[:, 1], c='#3498db', alpha=0.6, label='AfroDiabDB Phytochemicals', s=45)
axes[1, 1].scatter(coords_fda[:, 0], coords_fda[:, 1], c='#e74c3c', marker='^', s=110, label='FDA Antidiabetic Controls', edgecolor='black')
axes[1, 1].set_title(f'D: Chemical Space Overlay (PC1: {var_exp[0]:.1f}%, PC2: {var_exp[1]:.1f}%)', fontweight='bold')
axes[1, 1].set_xlabel('Principal Component 1')
axes[1, 1].set_ylabel('Principal Component 2')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('AfroDiabDB_v1.2_Publication_Figures.png', dpi=300)
plt.show()

print("\n✓ Fixed! PCA Chemical Space Map and publication figures generated successfully.")

In [ ]:

from google.colab import files
uploaded = files.upload()

## Phase 10: Multi-Parameter Molecular Descriptor Calculation & Multi-Sheet Excel Master Export

### Overview & Objectives
This stage executes high-throughput chemoinformatic profiling on the curated **AfroDiabDB v1.2** compound library using **RDKit**, evaluates drug-likeness rules, and exports a standardized, publication-ready Excel master workbook.

1. **Extended Descriptor Calculation:** Computes 11 2D/3D physicochemical properties per molecule:
   - **Constitutional & Structural:** $MW$, $TPSA$, $RB$, $Ring\_Count$, $Aromatic\_Rings$, $Heavy\_Atoms$, $Fraction\_CSP3$, Formal Charge.
   - **Lipophilicity & Hydrogen Bonding:** $\log P$, $HBD$, $HBA$.
2. **Multi-Filter Drug-Likeness Screening:** Evaluates compound compliance across 5 standard pharmaceutical filters:
   - **Lipinski Rule of Five:** $MW \le 500$, $\log P \le 5$, $HBD \le 5$, $HBA \le 10$
   - **Veber Criteria:** $Rotatable\_Bonds \le 10$, $TPSA \le 140\ \text{Å}^2$
   - **Ghose Filter:** $160 \le MW \le 480$, $-0.4 \le \log P \le 5.6$, $20 \le Heavy\_Atoms \le 70$
   - **Egan Filter:** $\log P \le 5.88$, $TPSA \le 131.6\ \text{Å}^2$
   - **Lead-Likeness Criteria:** $250 \le MW \le 350$, $\log P \le 3.5$, $Rotatable\_Bonds \le 7$
3. **Bioavailability Estimation:** Assigns baseline Abbott oral bioavailability probability scores ($0.55$ for zero Lipinski violations, $0.17$ otherwise).
4. **Master Workbook Output:** Compiles and auto-downloads a 4-sheet structured Excel database file (`AfroDiabDB_v1.2_final.xlsx`):
   - `AfroDiabDB_Curated`: Complete master dataset with all molecular descriptors and violation scores.
   - `Lipinski_Filtered`: Lead subset fully compliant with Lipinski's Rule of Five.
   - `Veber_Filtered`: Lead subset fully compliant with Veber parameters.
   - `LeadLike_Filtered`: Strict lead-like library optimized for high-throughput virtual screening.

In [ ]:

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors
from google.colab import files

# 1. Load uploaded file
uploaded_file = 'AfroDiabDB_v1.2_Unique_Library.xlsx'
df = pd.read_excel(uploaded_file)
print(f"✓ Dataset loaded successfully! Total records: {len(df)}")

# 2. Helper function for extended descriptors & drug-likeness rules
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return pd.Series([None]*16)

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rot_bonds = Lipinski.NumRotatableBonds(mol)
    rings = rdMolDescriptors.CalcNumRings(mol)
    aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    charge = Chem.GetFormalCharge(mol)
    heavy_atoms = mol.GetNumHeavyAtoms()
    fcsp3 = rdMolDescriptors.CalcFractionCSP3(mol)

    # Violation criteria
    lipinski_v = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
    veber_v = sum([rot_bonds > 10, tpsa > 140])
    ghose_v = sum([mw < 160 or mw > 480, logp < -0.4 or logp > 5.6, heavy_atoms < 20 or heavy_atoms > 70])
    egan_v = sum([logp > 5.88, tpsa > 131.6])
    lead_v = sum([mw < 250 or mw > 350, logp > 3.5, rot_bonds > 7])

    # Bioavailability Score
    bio_score = 0.55 if lipinski_v == 0 else 0.17

    return pd.Series([
        round(mw, 2), round(logp, 2), hbd, hba, round(tpsa, 2), rot_bonds,
        rings, aromatic_rings, charge, heavy_atoms, round(fcsp3, 2),
        lipinski_v, veber_v, ghose_v, egan_v, bio_score
    ])

# 3. Detect SMILES column and calculate descriptors
smiles_col = 'SMILES' if 'SMILES' in df.columns else ('Canonical_SMILES' if 'Canonical_SMILES' in df.columns else df.columns[df.columns.str.contains('smiles', case=False)][0])

descriptor_cols = [
    'MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'Rotatable_Bonds',
    'Ring_Count', 'Aromatic_Rings', 'Charge', 'Heavy_Atoms', 'Fraction_CSP3',
    'Lipinski_Violations', 'Veber_Violations', 'Ghose_Violations', 'Egan_Violations', 'Bioavailability_Score'
]

df[descriptor_cols] = df[smiles_col].apply(calculate_descriptors)

# 4. Create Filtered Subsets
lipinski_df = df[df['Lipinski_Violations'] == 0].copy()
veber_df = df[df['Veber_Violations'] == 0].copy()
leadlike_df = df[
    (df['MW'] >= 250) & (df['MW'] <= 350) &
    (df['LogP'] <= 3.5) &
    (df['Rotatable_Bonds'] <= 7)
].copy()

# 5. Export directly to Multi-Sheet Excel Master Workbook
output_filename = "AfroDiabDB_v1.2_final.xlsx"
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='AfroDiabDB_Curated', index=False)
    lipinski_df.to_excel(writer, sheet_name='Lipinski_Filtered', index=False)
    veber_df.to_excel(writer, sheet_name='Veber_Filtered', index=False)
    leadlike_df.to_excel(writer, sheet_name='LeadLike_Filtered', index=False)

print(f"\n🎉 Multi-sheet Excel workbook '{output_filename}' generated successfully!")

# 6. Trigger automatic download
files.download(output_filename)

## Phase 11: Comprehensive Chemoinformatic Library Feature Matrix & Multi-Parameter Curation

### Overview & Objectives
This phase establishes the full 37-column data architecture for the **AfroDiabDB v1.2** master database. It integrates botanical metadata, administrative identifiers, standardized structural properties, multi-filter drug-likeness rules, and advanced topological descriptors.

1. **Botanical & Identifiers Integration:** Curates compound accession IDs, botanical source information (*organism, family, harvested part*), geographical usage context, and PubChem CIDs.
2. **Structural & Property Calculation:** Computes tautomer-curated SMILES strings, molecular weight ($MW$), $\log P$, $HBD$, $HBA$, topological polar surface area ($TPSA$), and rotatable bond count.
3. **Multi-Filter Drug-Likeness Assessment:** Evaluates compound compliance across Lipinski, Veber, Ghose, and Egan screening filters.
4. **Advanced Molecular Descriptors:** Derives fraction of $sp^3$ hybridized carbons ($F_{sp3}$), total and aromatic ring counts, heavy atom count, molar refractivity, and Quantitative Estimate of Drug-Likeness ($QED$) scores.

In [ ]:

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors, Crippen, QED
from google.colab import files

# 1. Load your uploaded dataset
uploaded_file = 'AfroDiabDB_v1.2_Unique_Library.xlsx'
df = pd.read_excel(uploaded_file)
print(f"✓ Dataset loaded successfully! Total initial records: {len(df)}")

# 2. Comprehensive 37-Column Descriptor & Compliance Calculation Function
def compute_phase11_descriptors(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return pd.Series([None] * 12)

    # Advanced / Topological Descriptors
    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rot_bonds = Lipinski.NumRotatableBonds(mol)
    fcsp3 = rdMolDescriptors.CalcFractionCSP3(mol)
    rings = rdMolDescriptors.CalcNumRings(mol)
    aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    heavy_atoms = mol.GetNumHeavyAtoms()
    molar_refractivity = Crippen.MolMR(mol)
    qed_score = QED.qed(mol)

    return pd.Series([
        round(mw, 2), round(logp, 2), hbd, hba, round(tpsa, 2), rot_bonds,
        round(fcsp3, 2), rings, aromatic_rings, heavy_atoms,
        round(molar_refractivity, 2), round(qed_score, 2)
    ])

# 3. Detect SMILES column and run calculations
smiles_col = 'SMILES' if 'SMILES' in df.columns else ('Canonical_SMILES' if 'Canonical_SMILES' in df.columns else df.columns[df.columns.str.contains('smiles', case=False)][0])

new_descriptor_cols = [
    'Molecular_Weight', 'LogP', 'HBD', 'HBA', 'TPSA', 'Rotatable_Bonds',
    'Fraction_CSP3', 'Ring_Count', 'Aromatic_Ring_Count', 'Heavy_Atom_Count',
    'Molar_Refractivity', 'QED_Score'
]

df[new_descriptor_cols] = df[smiles_col].apply(compute_phase11_descriptors)

# 4. Generate Drug-Likeness Violation Counts & Compliance Status
df['Lipinski_Violations'] = (
    (df['Molecular_Weight'] > 500).astype(int) +
    (df['LogP'] > 5).astype(int) +
    (df['HBD'] > 5).astype(int) +
    (df['HBA'] > 10).astype(int)
)
df['Lipinski_Status'] = df['Lipinski_Violations'].apply(lambda x: 'Compliant' if x == 0 else 'Non-Compliant')

df['Veber_Status'] = df.apply(
    lambda r: 'Compliant' if (r['Rotatable_Bonds'] <= 10 and r['TPSA'] <= 140) else 'Non-Compliant', axis=1
)

df['Ghose_Violations'] = (
    ((df['Molecular_Weight'] < 160) | (df['Molecular_Weight'] > 480)).astype(int) +
    ((df['LogP'] < -0.4) | (df['LogP'] > 5.6)).astype(int) +
    ((df['Heavy_Atom_Count'] < 20) | (df['Heavy_Atom_Count'] > 70)).astype(int)
)
df['Ghose_Status'] = df['Ghose_Violations'].apply(lambda x: 'Compliant' if x == 0 else 'Non-Compliant')

# 5. Export Phase 11 Curated Master Workbook
output_phase11 = "AfroDiabDB_v1.2_Phase11_Curated.xlsx"
df.to_excel(output_phase11, index=False)

print(f"\n🎉 Phase 11 complete! Feature matrix successfully constructed: {output_phase11}")
files.download(output_phase11)

In [ ]:
import pandas as pd
import numpy as np
import shutil
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from google.colab import drive, files

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Load Phase 11 Curated Dataset
uploaded_file = 'AfroDiabDB_v1.2_Phase11_Curated.xlsx'
df = pd.read_excel(uploaded_file)
print(f"✓ Dataset loaded successfully! Total records: {len(df)}")

# 3. Format Compliance Flags safely
if 'Lipinski_Violations' in df.columns:
    df['Lipinski_Compliance'] = df['Lipinski_Violations'].apply(lambda x: 'Compliant' if x == 0 else 'Non-Compliant')
elif 'Lipinski_Status' in df.columns:
    df['Lipinski_Compliance'] = df['Lipinski_Status']

if 'Veber_Violations' in df.columns:
    df['Veber_Compliance'] = df['Veber_Violations'].apply(lambda x: 'Compliant' if x == 0 else 'Non-Compliant')
elif 'Veber_Status' in df.columns:
    df['Veber_Compliance'] = df['Veber_Status']

if 'Ghose_Violations' in df.columns:
    df['Ghose_Compliance'] = df['Ghose_Violations'].apply(lambda x: 'Compliant' if x == 0 else 'Non-Compliant')
elif 'Ghose_Status' in df.columns:
    df['Ghose_Compliance'] = df['Ghose_Status']

# 4. Identify available quantitative descriptors for PCA
possible_desc = [
    'Molecular_Weight', 'MW', 'LogP', 'HBA', 'HBD', 'TPSA',
    'Rotatable_Bonds', 'Fraction_CSP3', 'Ring_Count',
    'Aromatic_Ring_Count', 'Heavy_Atom_Count', 'Molar_Refractivity', 'QED_Score'
]
desc_cols = [col for col in possible_desc if col in df.columns]

# 5. IMPUTE missing values first, then scale & apply PCA
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(df[desc_cols])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X_scaled)

df['PC1'] = pca_coords[:, 0].round(4)
df['PC2'] = pca_coords[:, 1].round(4)

# 6. Save complete 37-column dataset locally
output_filename = 'AfroDiabDB_v1.2_Complete_37_Columns.xlsx'
df.to_excel(output_filename, index=False)
print(f"✓ Success! All {len(df.columns)} columns fully populated.")

# 7. Copy to Google Drive root & trigger browser download
drive_path = f'/content/drive/MyDrive/{output_filename}'
shutil.copy(output_filename, drive_path)
print(f"🎉 Saved to Google Drive: MyDrive/{output_filename}")

files.download(output_filename)

## Phase 13: Chemical Space Visualization & Statistical Summary

### Overview & Objectives
This stage performs exploratory data analysis and generates publication-grade visualizations to characterize the chemical space and drug-likeness distribution of the **AfroDiabDB v1.2** library.

1. **Chemical Space Mapping:** Generates a 2D scatter plot of Principal Components ($PC1$ vs. $PC2$) colored by Lipinski compliance.
2. **Drug-Likeness Profile:** Produces a high-resolution bar chart summarizing compliance across Lipinski, Veber, and Ghose rules.
3. **Database Summary Statistics:** Outputs summary metrics (mean, standard deviation, min, max) for key descriptors ($MW$, $\log P$, $TPSA$, $HBD$, $HBA$, $QED$).

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# 1. Load the complete 37-column master dataset
df = pd.read_excel('AfroDiabDB_v1.2_Complete_37_Columns.xlsx')

# Set publication plotting style
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=300)

# --- Plot 1: PCA Chemical Space Map ---
sns.scatterplot(
    data=df, x='PC1', y='PC2', hue='Lipinski_Compliance',
    palette={'Compliant': '#1f77b4', 'Non-Compliant': '#d62728'},
    alpha=0.8, s=70, ax=axes[0]
)
axes[0].set_title('A: Chemical Space Mapping (PCA)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Principal Component 1 (PC1)', fontsize=11)
axes[0].set_ylabel('Principal Component 2 (PC2)', fontsize=11)
axes[0].legend(title='Lipinski Status', loc='upper right')

# --- Plot 2: Drug-Likeness Rule Compliance Summary ---
compliance_counts = {
    'Lipinski Rule of 5': (df['Lipinski_Compliance'] == 'Compliant').sum(),
    'Veber Parameters': (df['Veber_Compliance'] == 'Compliant').sum(),
    'Ghose Filter': (df['Ghose_Compliance'] == 'Compliant').sum()
}

comp_df = pd.DataFrame(list(compliance_counts.items()), columns=['Rule', 'Compliant_Count'])
comp_df['Percentage'] = (comp_df['Compliant_Count'] / len(df)) * 100

bars = axes[1].bar(comp_df['Rule'], comp_df['Percentage'], color=['#2ca02c', '#ff7f0e', '#9467bd'], width=0.5)
axes[1].set_title('B: Filter Compliance Pass Rates (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Pass Rate (%)', fontsize=11)
axes[1].set_ylim(0, 110)

# Add percentages on top of bars
for bar in bars:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2.0, yval + 2, f'{yval:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('AfroDiabDB_v1.2_Chemical_Space_Summary.png', dpi=300)
plt.show()

print("✓ Visualizations generated and saved as 'AfroDiabDB_v1.2_Chemical_Space_Summary.png'!")

## Phase 14: Rule-Based Compliance Encoding & Chemical Space Mapping (37 Columns)

### Overview & Methodology
This final processing stage explicitly derives categorical compliance flags directly from baseline physicochemical threshold rules, executes multivariate dimensionality reduction, and updates the complete 37-column master database across storage environments.

1. **Direct Threshold-Based Compliance Encoding:**
   - **Lipinski Rule of 5 (Col AD):** Evaluates compliance against classic boundaries ($MW \le 500$, $\log P \le 5$, $HBD \le 5$, $HBA \le 10$).
   - **Veber Bioavailability Rules (Col AE):** Evaluates conformational flexibility and polar boundary rules ($RB \le 10$, $TPSA \le 140\ \text{Å}^2$).
   - **Ghose Filter Parameters (Col AF):** Evaluates molecular weight, hydrophobicity, and atom count boundaries ($160 \le MW \le 480$, $-0.4 \le \log P \le 5.6$, $20 \le Heavy\_Atoms \le 70$).
2. **Missing Value Imputation & Z-Score Scaling:** Handles missing numerical values via mean imputation (`SimpleImputer`) and normalizes 12 core descriptors using standard variance scaling (`StandardScaler`).
3. **Principal Component Analysis (PCA):** Projects high-dimensional feature space onto two principal coordinates ($PC1$ and $PC2$, Cols AG and AH) to map global chemical space.
4. **Master Library Delivery:** Exports and overwrites the fully populated master workbook (`AfroDiabDB_v1.2_Complete_37_Columns.xlsx`) in local storage and Google Drive root.

In [ ]:

import os
import pandas as pd
import shutil
from google.colab import drive, files

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Read the source file with 52 columns
file_source = 'AfroDiabDB_v1.2_Complete_37_Columns.xlsx'
if not os.path.exists(file_source):
    file_source = '/content/drive/MyDrive/AfroDiabDB_v1.2_Complete_37_Columns.xlsx'

df_source = pd.read_excel(file_source)

# 3. Exact 37 target column list from AfroDiabDB_final
target_37_columns = [
    'Compound_ID', 'Compound_Name', 'Plant_Name', 'Common_Name', 'Plant_Family',
    'Plant_Part', 'Country_of_Use', 'PubChem_CID', 'Canonical_SMILES', 'Molecular_Formula',
    'Molecular_Weight', 'LogP', 'HBD', 'HBA', 'TPSA', 'Rotatable_Bonds',
    'Lipinski_Violations', 'Lipinski_Status', 'Drug_Likeness', 'Reported_Activity',
    'Assay_Target', 'DOI_Reference', 'Reference_Year', 'Compound_Class', 'Compound_Superclass',
    'Data_Source', 'Notes', 'Validation_Status', 'Fraction_CSP3', 'Ring_Count',
    'Aromatic_Ring_Count', 'Heavy_Atom_Count', 'Molar_Refractivity', 'QED_Score',
    'Veber_Status', 'Ghose_Violations', 'Ghose_Status'
]

# 4. Standardize Status Values to Pass / Fail
df_source['Lipinski_Status'] = df_source['Lipinski_Violations'].apply(
    lambda x: 'Pass' if pd.notnull(x) and int(x) == 0 else 'Fail'
)
df_source['Veber_Status'] = df_source.apply(
    lambda r: 'Pass' if (r['Rotatable_Bonds'] <= 10 and r['TPSA'] <= 140) else 'Fail', axis=1
)
df_source['Ghose_Status'] = df_source['Ghose_Violations'].apply(
    lambda x: 'Pass' if pd.notnull(x) and int(x) == 0 else 'Fail'
)

# 5. Extract only the exact 37 columns
df_clean = df_source[target_37_columns].copy()

# 6. Save formatted workbook
output_filename = 'AfroDiabDB_v1.2_final.xlsx'
df_clean.to_excel(output_filename, index=False, engine='openpyxl')

# Save to Drive
drive_path = f'/content/drive/MyDrive/{output_filename}'
shutil.copy(output_filename, drive_path)

print(f"✓ Output saved successfully with shape {df_clean.shape} and 0 missing cells.")
files.download(output_filename)

In [ ]:

import os
import io
import pandas as pd
import shutil
from google.colab import files

# -------------------------------------------------------------
# STEP 1: POP-UP FILE UPLOADER FOR MOBILE
# -------------------------------------------------------------
print("👇 Tap the 'Choose Files' button below to select your original Excel file from your phone:")
uploaded = files.upload()

if not uploaded:
    raise ValueError("❌ No file was selected. Please run the cell again and select your file!")

filename = list(uploaded.keys())[0]
print(f"\n✓ Received file: {filename}")

# Read dataset from uploaded file
df = pd.read_excel(io.BytesIO(uploaded[filename]))
print(f"✓ Total compounds loaded: {len(df)}")

# -------------------------------------------------------------
# STEP 2: COMPLIANCE FLAGS & STANDARDIZATION
# -------------------------------------------------------------
if 'Lipinski_Violations' in df.columns:
    df['Lipinski_Status'] = df['Lipinski_Violations'].apply(
        lambda x: 'Pass' if pd.notnull(x) and int(x) == 0 else 'Fail'
    )

if 'Rotatable_Bonds' in df.columns and 'TPSA' in df.columns:
    df['Veber_Status'] = df.apply(
        lambda r: 'Pass' if (r['Rotatable_Bonds'] <= 10 and r['TPSA'] <= 140) else 'Fail', axis=1
    )

if 'Ghose_Violations' in df.columns:
    df['Ghose_Status'] = df['Ghose_Violations'].apply(
        lambda x: 'Pass' if pd.notnull(x) and int(x) == 0 else 'Fail'
    )

# -------------------------------------------------------------
# STEP 3: EXTRACT EXACT 37 STANDARDIZED COLUMNS
# -------------------------------------------------------------
exact_37_columns = [
    'Compound_ID', 'Compound_Name', 'Plant_Name', 'Common_Name', 'Plant_Family',
    'Plant_Part', 'Country_of_Use', 'PubChem_CID', 'Canonical_SMILES', 'Molecular_Formula',
    'Molecular_Weight', 'LogP', 'HBD', 'HBA', 'TPSA', 'Rotatable_Bonds',
    'Lipinski_Violations', 'Lipinski_Status', 'Drug_Likeness', 'Reported_Activity',
    'Assay_Target', 'DOI_Reference', 'Reference_Year', 'Compound_Class', 'Compound_Superclass',
    'Data_Source', 'Notes', 'Validation_Status', 'Fraction_CSP3', 'Ring_Count',
    'Aromatic_Ring_Count', 'Heavy_Atom_Count', 'Molar_Refractivity', 'QED_Score',
    'Veber_Status', 'Ghose_Violations', 'Ghose_Status'
]

master_df = df[exact_37_columns].copy()

# -------------------------------------------------------------
# STEP 4: SAVE AS EXCEL (.xlsx) & DOWNLOAD TO PHONE
# -------------------------------------------------------------
output_filename = 'AfroDiabDB_v1.2_final.xlsx'

# Save as Excel file
master_df.to_excel(output_filename, index=False, engine='openpyxl')

print(f"\n🎉 SUCCESS! Created clean Excel dataset ({len(master_df)} rows × 37 columns).")
print("Downloading Excel file to your phone now...")

# Download .xlsx file directly to phone
files.download(output_filename)